# Train Task 2: Multisensory Implicit Context

Sequential training: load a Task 1-trained CTRNN and continue training on Task 2.

In Task 2, **both modalities are always informative** and there is **no explicit context cue**. The network must infer which modality is currently relevant from reward feedback alone.

**Key behavioral signatures to look for:**
- **Congruent trials** (both cues agree): accuracy stays high through switches
- **Incongruent trials** (cues conflict): accuracy drops **below chance** immediately after a context switch (the network follows the wrong modality), then gradually recovers
- This below-chance dip is the hallmark of context inference from feedback

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.models import CTRNN
from src.tasks import Task2Session
from src.training import train_model, evaluate, run_session
from src.utils import set_seed, load_checkpoint

sns.set_style('whitegrid')
sns.set_context('notebook')

## Load Task 1 Checkpoint

Task 2 builds on the representations learned in Task 1. We use a lower learning rate to avoid destroying those representations.

In [ ]:
SEED = 42
HIDDEN_SIZE = 256

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')

# Load Task 1 model
model = CTRNN(
    input_size=8, hidden_size=HIDDEN_SIZE, output_size=2,
    dt=20.0, tau=100.0, sigma_rec=0.05, seed=SEED
).to(DEVICE)

checkpoint = load_checkpoint('../checkpoints/task1_final.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded Task 1 checkpoint (epoch {checkpoint["epoch"]})')

## Configuration for Task 2

In [ ]:
# --- Task 2 Hyperparameters ---
N_EPOCHS = 150           # Task 2 is harder, needs more epochs
LR = 5e-4               # Lower LR to preserve Task 1 representations
L1_LAMBDA = 1e-4
GRAD_CLIP = 1.0
N_SESSIONS_TRAIN = 8
N_SESSIONS_EVAL = 4
EVAL_EVERY = 5

# Task parameters
N_TRIALS = 300
BLOCK_SIZE = 50
CONGRUENT_RATIO = 0.5    # 50% congruent, 50% incongruent

set_seed(SEED)

## Train on Task 2

The model must now learn to:
1. Use reward feedback to infer which modality is currently relevant
2. Switch modality reliance when feedback indicates the context has changed

Training takes ~10-20 minutes depending on hardware.

In [ ]:
task_kwargs = {
    'n_trials': N_TRIALS,
    'block_size': BLOCK_SIZE,
    'congruent_ratio': CONGRUENT_RATIO,
}

history = train_model(
    model=model,
    task_class=Task2Session,
    task_kwargs=task_kwargs,
    device=DEVICE,
    n_epochs=N_EPOCHS,
    lr=LR,
    grad_clip=GRAD_CLIP,
    n_sessions_train=N_SESSIONS_TRAIN,
    n_sessions_eval=N_SESSIONS_EVAL,
    eval_every=EVAL_EVERY,
    l1_lambda=L1_LAMBDA,
    checkpoint_dir='../checkpoints',
    checkpoint_prefix='task2',
    seed=SEED,
    verbose=True
)

## Training Curves: Congruent vs Incongruent

The congruent accuracy should be consistently higher than incongruent, since congruent trials are correct regardless of which modality the network follows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], alpha=0.7, label='Train')
axes[0].plot(history['epoch'], history['eval_loss'], 'o-', label='Eval', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()

# Accuracy breakdown
axes[1].plot(history['train_accuracy'], alpha=0.5, color='gray', label='Train (overall)')
axes[1].plot(history['epoch'], history['eval_accuracy'], 'o-',
             label='Eval (overall)', markersize=3, color='black')

# Filter out None values for Task 2 specific metrics
cong_epochs = [e for e, v in zip(history['epoch'], history['eval_congruent_acc']) if v is not None]
cong_vals = [v for v in history['eval_congruent_acc'] if v is not None]
incong_vals = [v for v in history['eval_incongruent_acc'] if v is not None]

axes[1].plot(cong_epochs, cong_vals, 's-', label='Congruent', color='green', markersize=3)
axes[1].plot(cong_epochs, incong_vals, '^-', label='Incongruent', color='red', markersize=3)
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy by Trial Type')
axes[1].set_ylim(0.3, 1.05)
axes[1].legend(fontsize=8)

plt.suptitle('Task 2 Training', y=1.02)
plt.tight_layout()
plt.savefig('../figures/task2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Post-Switch Accuracy Dynamics

This is the **diagnostic plot** for implicit context inference. We align accuracy to context switch points and plot accuracy as a function of trials-since-switch.

**Expected pattern:**
- Congruent: high accuracy throughout (both modalities agree, so either works)
- Incongruent: drops **below 50%** right after switch (wrong modality), then recovers as the network infers the new context from feedback

In [ ]:
# Evaluate on many sessions for robust statistics
eval_result = evaluate(
    model, Task2Session, task_kwargs, DEVICE,
    n_sessions=16, seed=5555
)
print(f'Overall: {eval_result["accuracy"]:.3f}')
print(f'Congruent: {eval_result.get("congruent_accuracy", "N/A")}')
print(f'Incongruent: {eval_result.get("incongruent_accuracy", "N/A")}')

# Compute per-trial-in-block accuracy, split by congruence
cong_acc_by_pos = {}
incong_acc_by_pos = {}

for correct, meta in zip(eval_result['per_trial_accuracy'],
                         eval_result['per_trial_metadata']):
    pos = meta['trial_in_block']
    is_cong = meta.get('is_congruent', True)
    
    target = cong_acc_by_pos if is_cong else incong_acc_by_pos
    if pos not in target:
        target[pos] = []
    target[pos].append(int(correct))

# Compute mean accuracy at each position
positions = sorted(set(list(cong_acc_by_pos.keys()) + list(incong_acc_by_pos.keys())))
cong_means = [np.mean(cong_acc_by_pos.get(p, [0.5])) for p in positions]
incong_means = [np.mean(incong_acc_by_pos.get(p, [0.5])) for p in positions]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(positions, cong_means, 'o-', color='green', label='Congruent',
        markersize=3, linewidth=1.5)
ax.plot(positions, incong_means, 's-', color='red', label='Incongruent',
        markersize=3, linewidth=1.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
ax.axvline(0, color='black', linestyle='--', alpha=0.3, label='Switch point')

ax.set_xlabel('Trial Position Within Block (0 = just after switch)')
ax.set_ylabel('Accuracy')
ax.set_title('Post-Switch Accuracy Dynamics')
ax.legend()
ax.set_ylim(0.0, 1.05)

plt.tight_layout()
plt.savefig('../figures/task2_post_switch_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## Before vs After Switch Comparison

Compare accuracy in the last 10 trials of a block (well-adapted) vs the first 10 trials of the next block (just switched). Incongruent accuracy should be markedly lower after switches.

In [ ]:
# Split into pre-switch (last 10 of block) and post-switch (first 10 of block)
pre_cong, pre_incong = [], []
post_cong, post_incong = [], []

for correct, meta in zip(eval_result['per_trial_accuracy'],
                         eval_result['per_trial_metadata']):
    pos = meta['trial_in_block']
    is_cong = meta.get('is_congruent', True)
    
    if pos < 10:  # first 10 trials after switch
        if is_cong:
            post_cong.append(int(correct))
        else:
            post_incong.append(int(correct))
    elif pos >= BLOCK_SIZE - 10:  # last 10 trials before switch
        if is_cong:
            pre_cong.append(int(correct))
        else:
            pre_incong.append(int(correct))

labels = ['Pre-switch\nCongruent', 'Post-switch\nCongruent',
          'Pre-switch\nIncongruent', 'Post-switch\nIncongruent']
values = [np.mean(pre_cong) if pre_cong else 0,
          np.mean(post_cong) if post_cong else 0,
          np.mean(pre_incong) if pre_incong else 0,
          np.mean(post_incong) if post_incong else 0]
colors = ['green', 'lightgreen', 'red', 'lightsalmon']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=10)

ax.set_ylabel('Accuracy')
ax.set_title('Pre- vs Post-Switch Accuracy')
ax.set_ylim(0, 1.15)
ax.legend()

plt.tight_layout()
plt.savefig('../figures/task2_switch_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Task 2 Checkpoint

In [ ]:
print('Task 2 checkpoint saved at: ../checkpoints/task2_final.pt')
print(f'Overall accuracy: {eval_result["accuracy"]:.3f}')
print(f'Congruent accuracy: {eval_result.get("congruent_accuracy", "N/A")}')
print(f'Incongruent accuracy: {eval_result.get("incongruent_accuracy", "N/A")}')